In [1]:
import pandas as pd
import numpy as np

print("pandas:", pd.__version__)
print("numpy:", np.__version__)

pandas: 3.0.3
numpy: 2.4.6


In [2]:
pd.set_option('display.width', 150)

In [3]:
# Load the same intermediate artifact used in 03_modeling.ipynb.
train_bureau = pd.read_csv('../data/processed/train_bureau.csv')
print(train_bureau.shape)

(307511, 156)


In [4]:
train_bureau.head(2)

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,BUREAU_CREDIT_TYPE_NUNIQUE,BUREAU_DAYS_CREDIT_UPDATE_MEAN,BUREAU_DAYS_CREDIT_UPDATE_MAX,BUREAU_AMT_ANNUITY_SUM,BUREAU_AMT_ANNUITY_MEAN,BUREAU_CREDIT_ACTIVE_ACTIVE_COUNT,BUREAU_CREDIT_ACTIVE_BAD_DEBT_COUNT,BUREAU_CREDIT_ACTIVE_CLOSED_COUNT,BUREAU_CREDIT_ACTIVE_SOLD_COUNT,BUREAU_CREDIT_TYPE_MODE_COUNT
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,2.0,-499.875,-7.0,0.0,0.0,2.0,0.0,6.0,0.0,4.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,2.0,-816.000,-43.0,0.0,NaN,1.0,0.0,3.0,0.0,2.0


In [5]:
# Quick missing-value audit on train_bureau, freshly loaded in this notebook.
# Purpose: confirm missing counts before choosing which variables to bin first,
# without relying on memory from previous sessions.

missing_summary = pd.DataFrame({
    'dtype': train_bureau.dtypes,
    'n_missing': train_bureau.isnull().sum(),
    'pct_missing': (train_bureau.isnull().sum() / len(train_bureau) * 100).round(2)
})
missing_summary = missing_summary[missing_summary['n_missing'] > 0].sort_values('n_missing')

print("Total de colunas com missing:", len(missing_summary))
print()
print(missing_summary.head(20))

Total de colunas com missing: 98

                                     dtype  n_missing  pct_missing
DAYS_LAST_PHONE_CHANGE             float64          1         0.00
YEARS_LAST_PHONE_CHANGE            float64          1         0.00
CNT_FAM_MEMBERS                    float64          2         0.00
AMT_ANNUITY                        float64         12         0.00
AMT_GOODS_PRICE                    float64        278         0.09
EXT_SOURCE_2                       float64        660         0.21
OBS_60_CNT_SOCIAL_CIRCLE           float64       1021         0.33
OBS_30_CNT_SOCIAL_CIRCLE           float64       1021         0.33
DEF_60_CNT_SOCIAL_CIRCLE           float64       1021         0.33
DEF_30_CNT_SOCIAL_CIRCLE           float64       1021         0.33
NAME_TYPE_SUITE                        str       1292         0.42
AMT_REQ_CREDIT_BUREAU_HOUR         float64      41519        13.50
AMT_REQ_CREDIT_BUREAU_MON          float64      41519        13.50
AMT_REQ_CREDIT_BUREAU_QRT   

In [6]:
# Focused check on the EXT_SOURCE variables specifically, since they're
# the strongest predictors and the natural first candidates for the scorecard.

ext_source_cols = [c for c in train_bureau.columns if c.startswith('EXT_SOURCE')]
print(train_bureau[ext_source_cols].isnull().sum())

EXT_SOURCE_1    173378
EXT_SOURCE_2       660
EXT_SOURCE_3     60965
dtype: int64


In [7]:
# Bin EXT_SOURCE_2 into 10 quantile groups, with an explicit "Missing" bin
# for the 660 null values, instead of letting qcut silently drop them.

woe_df = pd.DataFrame({
    'EXT_SOURCE_2': train_bureau['EXT_SOURCE_2'],
    'TARGET': train_bureau['TARGET']
})

woe_df['bin'] = pd.qcut(woe_df['EXT_SOURCE_2'], q=10, duplicates='drop').astype('object')
woe_df.loc[woe_df['EXT_SOURCE_2'].isnull(), 'bin'] = 'Missing'

print(woe_df['bin'].value_counts(dropna=False))

bin
(0.646, 0.682]            30694
(0.34, 0.44]              30687
(0.566, 0.608]            30687
(-0.0009999183, 0.216]    30686
(0.216, 0.34]             30685
(0.722, 0.855]            30685
(0.512, 0.566]            30684
(0.44, 0.512]             30684
(0.608, 0.646]            30683
(0.682, 0.722]            30676
Missing                     660
Name: count, dtype: int64


In [8]:
# Calculate WoE and IV per bin (10 quantile bins + 1 explicit Missing bin).

total_good = (woe_df['TARGET'] == 0).sum()
total_bad = (woe_df['TARGET'] == 1).sum()

grouped = woe_df.groupby('bin', observed=True).agg(
    n_total=('TARGET', 'count'),
    n_bad=('TARGET', 'sum')
).reset_index()

grouped['n_good'] = grouped['n_total'] - grouped['n_bad']
grouped['pct_good'] = grouped['n_good'] / total_good
grouped['pct_bad'] = grouped['n_bad'] / total_bad
grouped['woe'] = np.log(grouped['pct_good'] / grouped['pct_bad'])
grouped['iv_component'] = (grouped['pct_good'] - grouped['pct_bad']) * grouped['woe']

iv_total = grouped['iv_component'].sum()

print(grouped[['bin', 'n_total', 'n_bad', 'pct_good', 'pct_bad', 'woe', 'iv_component']].round(4))
print()
print("IV total:", round(iv_total, 4))

                       bin  n_total  n_bad  pct_good  pct_bad     woe  iv_component
0   (-0.0009999183, 0.216]    30686   5631    0.0886   0.2268 -0.9397        0.1299
1            (0.216, 0.34]    30685   3706    0.0954   0.1493 -0.4474        0.0241
2             (0.34, 0.44]    30687   3056    0.0977   0.1231 -0.2307        0.0058
3            (0.44, 0.512]    30684   2566    0.0995   0.1034 -0.0384        0.0001
4           (0.512, 0.566]    30684   2278    0.1005   0.0918  0.0908        0.0008
5           (0.566, 0.608]    30687   2042    0.1013   0.0823  0.2086        0.0040
6           (0.608, 0.646]    30683   1794    0.1022   0.0723  0.3465        0.0104
7           (0.646, 0.682]    30694   1499    0.1033   0.0604  0.5367        0.0230
8           (0.682, 0.722]    30676   1289    0.1040   0.0519  0.6942        0.0361
9           (0.722, 0.855]    30685    912    0.1053   0.0367  1.0532        0.0722
10                 Missing      660     52    0.0022   0.0021  0.0264       

In [9]:
print(grouped[['woe']].round(4))

       woe
0  -0.9397
1  -0.4474
2  -0.2307
3  -0.0384
4   0.0908
5   0.2086
6   0.3465
7   0.5367
8   0.6942
9   1.0532
10  0.0264


In [10]:
def calculate_woe_iv(df, feature_col, target_col='TARGET', n_bins=10, min_bad_per_bin=100, smoothing=0.5):
    """
    Bin a numeric feature into quantile groups (plus an explicit 'Missing' bin),
    then calculate WoE and IV per bin. Applies additive (Laplace) smoothing to
    avoid division by zero when a bin has zero 'good' or 'bad' cases.
    """
    woe_df = pd.DataFrame({
        feature_col: df[feature_col],
        target_col: df[target_col]
    })

    woe_df['bin'] = pd.qcut(woe_df[feature_col], q=n_bins, duplicates='drop').astype('object')
    woe_df.loc[woe_df[feature_col].isnull(), 'bin'] = 'Missing'

    total_good = (woe_df[target_col] == 0).sum()
    total_bad = (woe_df[target_col] == 1).sum()

    grouped = woe_df.groupby('bin', observed=True).agg(
        n_total=(target_col, 'count'),
        n_bad=(target_col, 'sum')
    ).reset_index()

    grouped['n_good'] = grouped['n_total'] - grouped['n_bad']
    grouped['pct_good'] = (grouped['n_good'] + smoothing) / (total_good + smoothing * len(grouped))
    grouped['pct_bad'] = (grouped['n_bad'] + smoothing) / (total_bad + smoothing * len(grouped))
    grouped['woe'] = np.log(grouped['pct_good'] / grouped['pct_bad'])
    grouped['iv_component'] = (grouped['pct_good'] - grouped['pct_bad']) * grouped['woe']

    iv_total = grouped['iv_component'].sum()

    unstable_bins = grouped[grouped['n_bad'] < min_bad_per_bin]
    if len(unstable_bins) > 0:
        print(f"AVISO [{feature_col}]: {len(unstable_bins)} bin(s) com menos de {min_bad_per_bin} casos 'bad', WoE pode ser instável:")
        print(unstable_bins[['bin', 'n_bad']].to_string(index=False))
        print()

    return grouped[['bin', 'n_total', 'n_bad', 'pct_good', 'pct_bad', 'woe', 'iv_component']], iv_total

In [11]:
ext3_woe, ext3_iv = calculate_woe_iv(train_bureau, 'EXT_SOURCE_3')

print(ext3_woe.round(4))
print()
print("IV total (EXT_SOURCE_3):", round(ext3_iv, 4))

                   bin  n_total  n_bad  pct_good  pct_bad     woe  iv_component
0   (-0.000473, 0.228]    24701   4941    0.0699   0.1990 -1.0463        0.1351
1        (0.228, 0.33]    24744   3156    0.0764   0.1271 -0.5096        0.0259
2        (0.33, 0.408]    25057   2383    0.0802   0.0960 -0.1796        0.0028
3       (0.408, 0.476]    24689   1970    0.0804   0.0794  0.0127        0.0000
4       (0.476, 0.535]    24186   1494    0.0803   0.0602  0.2880        0.0058
5       (0.535, 0.592]    25392   1357    0.0850   0.0547  0.4416        0.0134
6       (0.592, 0.643]    24725   1173    0.0833   0.0473  0.5670        0.0204
7       (0.643, 0.694]    24745   1043    0.0838   0.0420  0.6907        0.0289
8       (0.694, 0.749]    23675    836    0.0808   0.0337  0.8747        0.0412
9       (0.749, 0.896]    24632    795    0.0843   0.0320  0.9678        0.0506
10             Missing    60965   5677    0.1956   0.2287 -0.1562        0.0052

IV total (EXT_SOURCE_3): 0.3293


In [12]:
def calculate_woe_iv_categorical(df, feature_col, target_col='TARGET', min_bad_per_bin=100, smoothing=0.5):
    """
    Calculate WoE and IV for a categorical feature, using each category
    as its own bin. Missing values (if any) get an explicit 'Missing' category.
    Applies additive (Laplace) smoothing to avoid division by zero when a
    category has zero 'good' or 'bad' cases.
    """
    woe_df = pd.DataFrame({
        feature_col: df[feature_col],
        target_col: df[target_col]
    })
    woe_df[feature_col] = woe_df[feature_col].astype('object')
    woe_df[feature_col] = woe_df[feature_col].fillna('Missing')

    total_good = (woe_df[target_col] == 0).sum()
    total_bad = (woe_df[target_col] == 1).sum()

    grouped = woe_df.groupby(feature_col, observed=True).agg(
        n_total=(target_col, 'count'),
        n_bad=(target_col, 'sum')
    ).reset_index().rename(columns={feature_col: 'bin'})

    grouped['n_good'] = grouped['n_total'] - grouped['n_bad']
    grouped['pct_good'] = (grouped['n_good'] + smoothing) / (total_good + smoothing * len(grouped))
    grouped['pct_bad'] = (grouped['n_bad'] + smoothing) / (total_bad + smoothing * len(grouped))
    grouped['woe'] = np.log(grouped['pct_good'] / grouped['pct_bad'])
    grouped['iv_component'] = (grouped['pct_good'] - grouped['pct_bad']) * grouped['woe']

    iv_total = grouped['iv_component'].sum()

    unstable_bins = grouped[grouped['n_bad'] < min_bad_per_bin]
    if len(unstable_bins) > 0:
        print(f"AVISO [{feature_col}]: {len(unstable_bins)} categoria(s) com menos de {min_bad_per_bin} casos 'bad', WoE pode ser instável:")
        print(unstable_bins[['bin', 'n_bad']].to_string(index=False))
        print()

    return grouped.sort_values('woe')[['bin', 'n_total', 'n_bad', 'pct_good', 'pct_bad', 'woe', 'iv_component']], iv_total

In [13]:
educ_woe, educ_iv = calculate_woe_iv_categorical(train_bureau, 'NAME_EDUCATION_TYPE')

print(educ_woe.round(4).to_string(index=False))
print()
print("IV total (NAME_EDUCATION_TYPE):", round(educ_iv, 4))

AVISO [NAME_EDUCATION_TYPE]: 1 categoria(s) com menos de 100 casos 'bad', WoE pode ser instável:
            bin  n_bad
Academic degree      3

                          bin  n_total  n_bad  pct_good  pct_bad     woe  iv_component
              Lower secondary     3816    417    0.0120   0.0168 -0.3353        0.0016
Secondary / secondary special   218391  19524    0.7035   0.7864 -0.1114        0.0092
            Incomplete higher    10277    872    0.0333   0.0351 -0.0547        0.0001
             Higher education    74863   4009    0.2506   0.1615  0.4396        0.0392
              Academic degree      164      3    0.0006   0.0001  1.3993        0.0006

IV total (NAME_EDUCATION_TYPE): 0.0507


In [14]:
# Merge 'Academic degree' (only 164 clients, unstable WoE) into 'Higher education',
# the closest category in both WoE and business meaning (both represent
# post-secondary education).

train_bureau['NAME_EDUCATION_TYPE_BINNED'] = train_bureau['NAME_EDUCATION_TYPE'].replace(
    {'Academic degree': 'Higher education'}
)

print(train_bureau['NAME_EDUCATION_TYPE_BINNED'].value_counts())

NAME_EDUCATION_TYPE_BINNED
Secondary / secondary special    218391
Higher education                  75027
Incomplete higher                 10277
Lower secondary                    3816
Name: count, dtype: int64


C:\Users\vitor\AppData\Local\Temp\ipykernel_3000\3584352028.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_bureau['NAME_EDUCATION_TYPE_BINNED'] = train_bureau['NAME_EDUCATION_TYPE'].replace(


In [15]:
educ_woe_v2, educ_iv_v2 = calculate_woe_iv_categorical(train_bureau, 'NAME_EDUCATION_TYPE_BINNED')

print(educ_woe_v2.round(4).to_string(index=False))
print()
print("IV total (NAME_EDUCATION_TYPE_BINNED):", round(educ_iv_v2, 4))
print("Comparação, IV original (5 categorias):", round(educ_iv, 4))

                          bin  n_total  n_bad  pct_good  pct_bad     woe  iv_component
              Lower secondary     3816    417    0.0120   0.0168 -0.3353        0.0016
Secondary / secondary special   218391  19524    0.7035   0.7864 -0.1114        0.0092
            Incomplete higher    10277    872    0.0333   0.0351 -0.0547        0.0001
             Higher education    75027   4012    0.2512   0.1616  0.4411        0.0395

IV total (NAME_EDUCATION_TYPE_BINNED): 0.0505
Comparação, IV original (5 categorias): 0.0507


In [16]:
ext1_woe, ext1_iv = calculate_woe_iv(train_bureau, 'EXT_SOURCE_1')

print(ext1_woe.round(4).to_string(index=False))
print()
print("IV total (EXT_SOURCE_1):", round(ext1_iv, 4))

                          bin  n_total  n_bad  pct_good  pct_bad     woe  iv_component
(0.013600000000000001, 0.213]    13414   2356    0.0391   0.0949 -0.8863        0.0494
               (0.213, 0.296]    13413   1555    0.0419   0.0626 -0.4010        0.0083
               (0.296, 0.369]    13414   1220    0.0431   0.0492 -0.1306        0.0008
               (0.369, 0.438]    13412   1124    0.0435   0.0453 -0.0410        0.0001
               (0.438, 0.506]    13415    898    0.0443   0.0362  0.2019        0.0016
               (0.506, 0.573]    13412    808    0.0446   0.0326  0.3143        0.0038
                (0.573, 0.64]    13413    689    0.0450   0.0278  0.4830        0.0083
                 (0.64, 0.71]    13413    588    0.0454   0.0237  0.6493        0.0141
                (0.71, 0.787]    13413    471    0.0458   0.0190  0.8801        0.0236
               (0.787, 0.963]    13414    345    0.0462   0.0139  1.2008        0.0388
                      Missing   173378  147

In [17]:
train_bureau.head(2)

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,BUREAU_DAYS_CREDIT_UPDATE_MEAN,BUREAU_DAYS_CREDIT_UPDATE_MAX,BUREAU_AMT_ANNUITY_SUM,BUREAU_AMT_ANNUITY_MEAN,BUREAU_CREDIT_ACTIVE_ACTIVE_COUNT,BUREAU_CREDIT_ACTIVE_BAD_DEBT_COUNT,BUREAU_CREDIT_ACTIVE_CLOSED_COUNT,BUREAU_CREDIT_ACTIVE_SOLD_COUNT,BUREAU_CREDIT_TYPE_MODE_COUNT,NAME_EDUCATION_TYPE_BINNED
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,-499.875,-7.0,0.0,0.0,2.0,0.0,6.0,0.0,4.0,Secondary / secondary special
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,-816.000,-43.0,0.0,NaN,1.0,0.0,3.0,0.0,2.0,Higher education


In [18]:
exclude_cols = ['SK_ID_CURR', 'TARGET']
all_candidates = [c for c in train_bureau.columns if c not in exclude_cols]

iv_results = []

for col in all_candidates:
    try:
        if pd.api.types.is_numeric_dtype(train_bureau[col]):
            _, iv = calculate_woe_iv(train_bureau, col)
            var_type = 'numeric'
        else:
            _, iv = calculate_woe_iv_categorical(train_bureau, col)
            var_type = 'categorical'
        iv_results.append({'variable': col, 'type': var_type, 'iv': round(iv, 4)})
    except Exception as e:
        iv_results.append({'variable': col, 'type': 'ERROR', 'iv': None})
        print(f"Erro em {col}: {e}")

iv_summary = pd.DataFrame(iv_results).sort_values('iv', ascending=False)
print(iv_summary.to_string(index=False))

AVISO [CODE_GENDER]: 1 categoria(s) com menos de 100 casos 'bad', WoE pode ser instável:
bin  n_bad
XNA      0

AVISO [AMT_ANNUITY]: 1 bin(s) com menos de 100 casos 'bad', WoE pode ser instável:
    bin  n_bad
Missing      0

AVISO [AMT_GOODS_PRICE]: 1 bin(s) com menos de 100 casos 'bad', WoE pode ser instável:
    bin  n_bad
Missing     21

AVISO [NAME_TYPE_SUITE]: 3 categoria(s) com menos de 100 casos 'bad', WoE pode ser instável:
            bin  n_bad
Group of people     23
        Missing     70
        Other_A     76

AVISO [NAME_INCOME_TYPE]: 4 categoria(s) com menos de 100 casos 'bad', WoE pode ser instável:
            bin  n_bad
    Businessman      0
Maternity leave      2
        Student      0
     Unemployed      8

AVISO [NAME_EDUCATION_TYPE]: 1 categoria(s) com menos de 100 casos 'bad', WoE pode ser instável:
            bin  n_bad
Academic degree      3

AVISO [NAME_FAMILY_STATUS]: 1 categoria(s) com menos de 100 casos 'bad', WoE pode ser instável:
    bin  n_bad
Unkno

In [19]:
import re

def get_base_concept(var_name):
    """Strip common redundant suffixes/prefixes to group near-duplicate variables."""
    name = re.sub(r'_(AVG|MODE|MEDI)$', '', var_name)
    name = re.sub(r'^DAYS_', 'AGE_MARKER_', name)
    name = re.sub(r'^YEARS_', 'AGE_MARKER_', name)
    if name == 'AGE_YEARS':
        name = 'AGE_MARKER_BIRTH'
    return name

iv_summary['base_concept'] = iv_summary['variable'].apply(get_base_concept)

# Show only concepts with more than one variable mapped to them,
# so you can visually confirm the grouping makes sense before any cut.
group_sizes = iv_summary.groupby('base_concept').size()
groups_with_duplicates = group_sizes[group_sizes > 1].index

for concept in sorted(groups_with_duplicates):
    subset = iv_summary[iv_summary['base_concept'] == concept].sort_values('iv', ascending=False)
    print(f"=== {concept} ===")
    print(subset[['variable', 'iv']].to_string(index=False))
    print()

=== AGE_MARKER_BEGINEXPLUATATION ===
                    variable     iv
YEARS_BEGINEXPLUATATION_MEDI 0.0292
 YEARS_BEGINEXPLUATATION_AVG 0.0291
YEARS_BEGINEXPLUATATION_MODE 0.0290

=== AGE_MARKER_BIRTH ===
  variable     iv
 AGE_YEARS 0.0842
DAYS_BIRTH 0.0842

=== AGE_MARKER_BUILD ===
        variable     iv
YEARS_BUILD_MODE 0.0177
YEARS_BUILD_MEDI 0.0176
 YEARS_BUILD_AVG 0.0176

=== AGE_MARKER_EMPLOYED ===
      variable     iv
YEARS_EMPLOYED 0.1113
 DAYS_EMPLOYED 0.1111

=== AGE_MARKER_ID_PUBLISH ===
        variable     iv
 DAYS_ID_PUBLISH 0.0384
YEARS_ID_PUBLISH 0.0384

=== AGE_MARKER_LAST_PHONE_CHANGE ===
               variable     iv
 DAYS_LAST_PHONE_CHANGE 0.0467
YEARS_LAST_PHONE_CHANGE 0.0463

=== AGE_MARKER_REGISTRATION ===
          variable     iv
 DAYS_REGISTRATION 0.0269
YEARS_REGISTRATION 0.0269

=== APARTMENTS ===
       variable     iv
 APARTMENTS_AVG 0.0323
APARTMENTS_MEDI 0.0320
APARTMENTS_MODE 0.0316

=== BASEMENTAREA ===
         variable     iv
 BASEMENTAREA_AVG 

In [20]:
# Keep only the highest-IV variable per concept group, then select top 40 overall.

deduped = iv_summary.sort_values('iv', ascending=False).drop_duplicates(subset='base_concept', keep='first')

print(f"Após dedupe: {len(deduped)} de {len(iv_summary)} variáveis")
print()

top_40 = deduped.sort_values('iv', ascending=False).head(40)
print(top_40[['variable', 'type', 'iv']].to_string(index=False))

Após dedupe: 122 de 155 variáveis

                         variable        type     iv
                     EXT_SOURCE_3     numeric 0.3293
                     EXT_SOURCE_2     numeric 0.3063
                     EXT_SOURCE_1     numeric 0.1508
          BUREAU_DAYS_CREDIT_MEAN     numeric 0.1227
                   YEARS_EMPLOYED     numeric 0.1113
   BUREAU_DAYS_CREDIT_UPDATE_MEAN     numeric 0.0934
                  AMT_GOODS_PRICE     numeric 0.0919
                        AGE_YEARS     numeric 0.0842
                  OCCUPATION_TYPE categorical 0.0828
           BUREAU_DAYS_CREDIT_MIN     numeric 0.0762
                ORGANIZATION_TYPE categorical 0.0731
  BUREAU_DAYS_CREDIT_ENDDATE_MEAN     numeric 0.0713
    BUREAU_DAYS_ENDDATE_FACT_MEAN     numeric 0.0702
                 NAME_INCOME_TYPE categorical 0.0584
  BUREAU_AMT_CREDIT_SUM_DEBT_MEAN     numeric 0.0532
              NAME_EDUCATION_TYPE categorical 0.0507
       NAME_EDUCATION_TYPE_BINNED categorical 0.0505
BUREAU_CRED

In [21]:
# Systematic rule: if a "_BINNED" version of a variable exists in the list,
# it supersedes the original (it represents a manual fix, like merging an
# unstable category). Drop the original, keep the binned version.

def resolve_binned_overrides(df):
    variables = set(df['variable'])
    to_drop = []
    for var in variables:
        base_name = var.replace('_BINNED', '')
        if var.endswith('_BINNED') and base_name in variables:
            to_drop.append(base_name)
    return df[~df['variable'].isin(to_drop)]

deduped = iv_summary.sort_values('iv', ascending=False).drop_duplicates(subset='base_concept', keep='first')
deduped = resolve_binned_overrides(deduped)

print(f"Após dedupe: {len(deduped)} de {len(iv_summary)} variáveis")
print()

top_40 = deduped.sort_values('iv', ascending=False).head(40)
print(top_40[['variable', 'type', 'iv']].to_string(index=False))

Após dedupe: 121 de 155 variáveis

                         variable        type     iv
                     EXT_SOURCE_3     numeric 0.3293
                     EXT_SOURCE_2     numeric 0.3063
                     EXT_SOURCE_1     numeric 0.1508
          BUREAU_DAYS_CREDIT_MEAN     numeric 0.1227
                   YEARS_EMPLOYED     numeric 0.1113
   BUREAU_DAYS_CREDIT_UPDATE_MEAN     numeric 0.0934
                  AMT_GOODS_PRICE     numeric 0.0919
                        AGE_YEARS     numeric 0.0842
                  OCCUPATION_TYPE categorical 0.0828
           BUREAU_DAYS_CREDIT_MIN     numeric 0.0762
                ORGANIZATION_TYPE categorical 0.0731
  BUREAU_DAYS_CREDIT_ENDDATE_MEAN     numeric 0.0713
    BUREAU_DAYS_ENDDATE_FACT_MEAN     numeric 0.0702
                 NAME_INCOME_TYPE categorical 0.0584
  BUREAU_AMT_CREDIT_SUM_DEBT_MEAN     numeric 0.0532
       NAME_EDUCATION_TYPE_BINNED categorical 0.0505
BUREAU_CREDIT_ACTIVE_ACTIVE_COUNT     numeric 0.0504
   BUREAU_A

In [22]:
top_40_vars = top_40['variable'].tolist()

woe_details = {}
unstable_report = []

for var in top_40_vars:
    if pd.api.types.is_numeric_dtype(train_bureau[var]):
        result, iv = calculate_woe_iv(train_bureau, var)
    else:
        result, iv = calculate_woe_iv_categorical(train_bureau, var)
    woe_details[var] = result
    unstable = result[result['n_bad'] < 100]
    if len(unstable) > 0:
        unstable_report.append({'variable': var, 'n_unstable_bins': len(unstable)})

unstable_df = pd.DataFrame(unstable_report)
print(unstable_df.to_string(index=False))

AVISO [EXT_SOURCE_2]: 1 bin(s) com menos de 100 casos 'bad', WoE pode ser instável:
    bin  n_bad
Missing     52

AVISO [AMT_GOODS_PRICE]: 1 bin(s) com menos de 100 casos 'bad', WoE pode ser instável:
    bin  n_bad
Missing     21

AVISO [OCCUPATION_TYPE]: 4 categoria(s) com menos de 100 casos 'bad', WoE pode ser instável:
          bin  n_bad
     HR staff     36
     IT staff     34
Realty agents     59
  Secretaries     92

AVISO [ORGANIZATION_TYPE]: 27 categoria(s) com menos de 100 casos 'bad', WoE pode ser instável:
                bin  n_bad
        Advertising     35
           Cleaning     29
            Culture     21
        Electricity     63
          Emergency     40
              Hotel     62
  Industry: type 10      7
  Industry: type 12     14
  Industry: type 13      9
   Industry: type 2     33
   Industry: type 4     89
   Industry: type 5     41
   Industry: type 6      8
   Industry: type 8      3
          Insurance     34
     Legal Services     24
             

In [23]:
def collapse_rare_categories(df, feature_col, target_col='TARGET', min_bad=100):
    """
    Relabel categories with fewer than min_bad 'bad' cases as 'Other_grouped'
    (a name unlikely to collide with existing categories). If the resulting
    grouped bucket is still below min_bad, merge it into the nearest-WoE
    'large' category instead of leaving it unstable.
    """
    series = df[feature_col].astype('object').fillna('Missing')
    bad_counts = df.groupby(series)[target_col].sum()
    rare_categories = bad_counts[bad_counts < min_bad].index.tolist()

    new_series = series.replace(rare_categories, 'Other_grouped')

    # Check if the grouped bucket itself is still below threshold
    new_bad_counts = df.groupby(new_series)[target_col].sum()
    if 'Other_grouped' in new_bad_counts.index and new_bad_counts['Other_grouped'] < min_bad:
        temp_df = pd.DataFrame({feature_col: new_series, target_col: df[target_col]})
        temp_result, _ = calculate_woe_iv_categorical(temp_df, feature_col, target_col, min_bad_per_bin=999999)
        grouped_woe = temp_result.loc[temp_result['bin'] == 'Other_grouped', 'woe'].values[0]
        large_bins = temp_result[(temp_result['bin'] != 'Other_grouped') & (temp_result['n_bad'] >= min_bad)].copy()
        large_bins['woe_diff'] = (large_bins['woe'] - grouped_woe).abs()
        nearest_large = large_bins.sort_values('woe_diff').iloc[0]['bin']
        new_series = new_series.replace('Other_grouped', nearest_large)
        print(f"'{feature_col}': Other_grouped ainda instável, fundido com '{nearest_large}'")

    return new_series

In [24]:
categorical_to_fix = ['OCCUPATION_TYPE', 'NAME_INCOME_TYPE', 'ORGANIZATION_TYPE',
                       'WALLSMATERIAL_MODE']

for col in categorical_to_fix:
    new_col = f'{col}_BINNED'
    train_bureau[new_col] = collapse_rare_categories(train_bureau, col)
    result, iv = calculate_woe_iv_categorical(train_bureau, new_col)
    print(f"=== {new_col} (IV: {round(iv, 4)}) ===")
    print(result[['bin', 'n_total', 'n_bad', 'woe']].to_string(index=False))
    print()

C:\Users\vitor\AppData\Local\Temp\ipykernel_3000\4104350587.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_bureau[new_col] = collapse_rare_categories(train_bureau, col)
C:\Users\vitor\AppData\Local\Temp\ipykernel_3000\4104350587.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_bureau[new_col] = collapse_rare_categories(train_bureau, col)


=== OCCUPATION_TYPE_BINNED (IV: 0.0828) ===
                  bin  n_total  n_bad       woe
   Low-skill Laborers     2093    359 -0.858431
              Drivers    18603   2107 -0.374546
 Waiters/barmen staff     1348    152 -0.372200
       Security staff     6721    722 -0.315478
             Laborers    55186   5838 -0.297758
        Cooking staff     5946    621 -0.284066
          Sales staff    32102   3092 -0.193514
       Cleaning staff     4653    447 -0.191482
        Other_grouped     3145    221  0.148265
       Medicine staff     8537    572  0.200671
Private service staff     2652    175  0.215175
              Missing    96391   6278  0.231747
           Core staff    27570   1738  0.266419
             Managers    21371   1328  0.281663
High skill tech staff    11380    701  0.290669
          Accountants     9813    474  0.547555

AVISO [NAME_INCOME_TYPE]: 5 categoria(s) com menos de 999999 casos 'bad', WoE pode ser instável:
                 bin  n_bad
Commercial ass

C:\Users\vitor\AppData\Local\Temp\ipykernel_3000\4104350587.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_bureau[new_col] = collapse_rare_categories(train_bureau, col)
C:\Users\vitor\AppData\Local\Temp\ipykernel_3000\4104350587.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_bureau[new_col] = collapse_rare_categories(train_bureau, col)
